<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/15_GES_Aware_Genomic_RAG_Cell_7C8_Blinded_Reviewer_and_Adjudication_Packet_Materialization_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {ROOT}')

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact hashes, and fail-closed output package

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

NOTEBOOK_NAME = (
    '15_GES_Aware_Genomic_RAG_Cell_7C8_'
    'Blinded_Reviewer_and_Adjudication_Packet_Materialization.ipynb'
)
CELL_ID = '7C8'
STAGE = '7C'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80
EXPECTED_CONTEXT_ROWS = 2_400
EXPECTED_RUBRIC_ROWS = 640
EXPECTED_RUBRICS_PER_QUESTION = 8
EXPECTED_REVIEWERS = ['REVIEWER-1', 'REVIEWER-2']
EXPECTED_REVIEW_ASSIGNMENTS = 2_880
EXPECTED_REVIEWER_RUBRIC_ROWS = 23_040

EXPECTED_CELL_7C7_TERMINAL_DECISION = (
    'PASS_STAGE7C7_COMPLETE_CELL7C6_V2_GENERATION_PACKAGE_AND_FROZEN_CELL7B2_7B3_'
    'EVALUATION_DESIGN_REVERIFIED_CHECKSUM_PROTECTED_CELL7C8_BLINDED_REVIEW_PACKET_'
    'MATERIALIZATION_ONLY_AUTHORIZED_ANSWER_KEYS_AND_RUBRICS_MAY_BE_OPENED_IN_7C8_'
    'NO_CONDITION_UNBLINDING_SCORE_BEARING_ARTIFACTS_RUN_AGGREGATION_RAG_METRICS_'
    'BOOTSTRAP_OR_ARM_COMPARISON'
)

EXPECTED_CELL_7C7_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C8_BLINDED_REVIEW_PACKET_MATERIALIZATION_ONLY_'
    'FROM_1440_FROZEN_STRUCTURED_RESPONSES_80_STRUCTURED_ANSWER_KEYS_640_RUBRIC_'
    'ASSIGNMENTS_AND_2400_SCORE_BLIND_CONTEXT_ROWS_TWO_INDEPENDENT_REVIEWERS_'
    'THIRD_ADJUDICATOR_NO_CONDITION_UNBLINDING_SCORE_BEARING_ARTIFACTS_RUN_AGGREGATION_'
    'RAG_METRICS_BOOTSTRAP_OR_ARM_COMPARISON'
)

CELL_7C7_AUTH_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c7_blinded_evaluation_packet_authorization_v1'
)
CELL_7C7_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c7_blinded_evaluation_packet_authorization_v1'
)

CELL_7C7 = OrderedDict([
    ('authorization', {
        'path': CELL_7C7_AUTH_DIR / 'cell_7c7_stage7c_cell7c8_blinded_evaluation_packet_authorization_v1.json',
        'sha256': 'c5aa662e23fffc88251e0f5f5ac77b2ca791a93d19704e5d1a0886168607df3f',
    }),
    ('input_inventory', {
        'path': CELL_7C7_AUTH_DIR / 'cell_7c7_authorized_blinded_evaluation_input_inventory_v1.csv',
        'sha256': '85e12035cc5219f8e4807171a859b2e2aadbd2c9e8f888f378159a71c966b250',
    }),
    ('protocol_snapshot', {
        'path': CELL_7C7_AUTH_DIR / 'cell_7c7_frozen_blinded_evaluation_protocol_snapshot_v1.json',
        'sha256': '3206eb76cb6ffc483f007ed377c38cf213c777c1f6771462f308245e381271cc',
    }),
    ('qc', {
        'path': CELL_7C7_QC_DIR / 'cell_7c7_blinded_evaluation_packet_authorization_qc_v1.json',
        'sha256': '22fec31af2a05856e08c2406f6e00977eec57b0e94dde8c56c2b5e99c946737a',
    }),
    ('manifest', {
        'path': CELL_7C7_AUTH_DIR / 'cell_7c7_blinded_evaluation_packet_authorization_manifest_v1.json',
        'sha256': 'e4dabd9a1eb7715d3f89e44137f369688650f3a812ef6edcfdfef1d7a1be894e',
    }),
])

CELL_7C6_STRUCTURED = {
    'filename': 'cell_7c6_v2_structured_response_outputs_v1.parquet',
    'sha256': '267edad192f41ab32e211ee43b083ca167e6eb6f9d3e63008822e39b57317a46',
}
CELL_7C4_CONTEXT = {
    'filename': 'cell_7c4_score_blind_prompt_context_inventory_v1.parquet',
    'sha256': 'd23b604dac645c158cd1b95cc9cd564fb6b9447279756b6cae589b5572dd4623',
}
CELL_7B3_ANSWER = {
    'filename': 'cell_7b3_structured_answer_keys_v1.parquet',
    'sha256': 'bccf3691336ed8e21a4e7b2876fcf6883ee9a8447938a0edf61d41fc5e0d0332',
}
CELL_7B3_PRIMARY_QUESTIONS = {
    'filename': 'cell_7b3_primary_question_set_v1.csv',
    'sha256': 'c76e81952fcc6a698866b64da7b7daabeb281b7b9d10e17873596095b69d95df',
}
CELL_7B3_RUBRIC = {
    'filename': 'cell_7b3_question_rubric_assignment_inventory_v1.csv',
    'sha256': '4185ad173c5e3e2ecea31268b8d833ba111839f5ddd1b96c3fda083e01afcba9',
}
CELL_7B2_MATERIALIZATION_SHA256 = '631b51556b97b3cf0db6d0989fa380b8fe3b226472de90c89f57eb0cae527da2'
CELL_7B2_ADJUDICATION_SHA256 = 'f79ed83b5e5a614815919f6d2b10efd01947c1512b02502e258820431a4c3ef0'

EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c8_blinded_reviewer_packet_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c8_blinded_reviewer_packet_v1'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c8_blinded_reviewer_packet_v1'
)

OUTPUTS = OrderedDict([
    ('review_packet',
     EXEC_DIR / 'cell_7c8_blinded_review_packet_v1.parquet'),
    ('reviewer_assignments',
     EXEC_DIR / 'cell_7c8_two_reviewer_assignment_inventory_v1.csv'),
    ('rubric_scoring_template',
     EXEC_DIR / 'cell_7c8_reviewer_rubric_scoring_template_v1.csv'),
    ('atomic_claim_template',
     EXEC_DIR / 'cell_7c8_reviewer_atomic_claim_annotation_template_v1.csv'),
    ('adjudicator_template',
     EXEC_DIR / 'cell_7c8_third_adjudicator_disagreement_template_v1.csv'),
    ('internal_routing_map',
     CONFIG_DIR / 'cell_7c8_internal_blinded_review_routing_map_v1.parquet'),
    ('reviewer_instructions',
     CONFIG_DIR / 'cell_7c8_blinded_reviewer_instructions_v1.json'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c8_verified_input_inventory_v1.csv'),
    ('execution_report',
     QC_DIR / 'cell_7c8_blinded_reviewer_packet_execution_report_v1.json'),
    ('qc',
     QC_DIR / 'cell_7c8_blinded_reviewer_packet_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c8_blinded_reviewer_packet_manifest_v1.json'),
])

for directory in (EXEC_DIR, QC_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C8 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Execution directory: {EXEC_DIR}')
print(f'QC directory       : {QC_DIR}')
print(f'Config directory   : {CONFIG_DIR}')

Execution directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/rag_execution/stage7_rag/cell_7c8_blinded_reviewer_packet_v1
QC directory       : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c8_blinded_reviewer_packet_v1
Config directory   : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c8_blinded_reviewer_packet_v1


## 2. Checksum, serialization, canonical JSON, and artifact-location helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')

def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar format: {path}')
    return token.lower()

def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )

def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }

def locate_exact(filename: str, expected_sha256: str) -> Path:
    candidates = [path for path in ROOT.rglob(filename) if path.is_file()]
    matches = [path for path in candidates if sha256_file(path) == expected_sha256]
    if len(matches) != 1:
        raise RuntimeError(
            f'Expected exactly one checksum-matching {filename}; found {len(matches)}.'
        )
    return matches[0]

def locate_config_hash(expected_sha256: str) -> Path:
    candidates = []
    for base in [ROOT / 'configs', ROOT / 'outputs' / 'quality_checks']:
        if base.exists():
            candidates.extend(
                path for path in base.rglob('*')
                if path.is_file() and not path.name.endswith('.sha256')
            )
    matches = [path for path in candidates if sha256_file(path) == expected_sha256]
    if len(matches) != 1:
        raise RuntimeError(
            f'Expected exactly one protocol/config artifact with SHA-256 {expected_sha256}; '
            f'found {len(matches)}.'
        )
    return matches[0]

def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))

def to_native(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, np.generic):
        return to_native(value.item())
    if isinstance(value, np.ndarray):
        return [to_native(v) for v in value.tolist()]
    if isinstance(value, (list, tuple)):
        return [to_native(v) for v in value]
    if isinstance(value, dict):
        return {str(k): to_native(v) for k, v in value.items()}
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return str(value)

def canonical_json(payload: Any) -> str:
    return json.dumps(
        to_native(payload),
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':'),
        allow_nan=False,
    )

def stable_write_json(path: Path, payload: Any) -> str:
    text = json.dumps(
        to_native(payload),
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    ) + chr(10)
    path.write_text(text, encoding='utf-8')
    return sha256_file(path)

def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)

def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    frame.to_parquet(path, index=False, engine='pyarrow', compression='zstd')
    return sha256_file(path)

def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )

def row_to_json(row: pd.Series) -> str:
    return canonical_json({column: to_native(row[column]) for column in row.index})

with tempfile.TemporaryDirectory(prefix='cell_7c8_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify Cell 7C7 authorization and exact authorized inputs

In [4]:
verified_inputs = []

for key, spec in CELL_7C7.items():
    record = verify_exact_artifact(
        f'cell_7c7_{key}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C7'
    verified_inputs.append(record)

authorization_7c7 = load_json(CELL_7C7['authorization']['path'])
manifest_7c7 = load_json(CELL_7C7['manifest']['path'])
qc_7c7 = load_json(CELL_7C7['qc']['path'])
protocol_snapshot_7c7 = load_json(CELL_7C7['protocol_snapshot']['path'])

if manifest_7c7.get('terminal_decision') != EXPECTED_CELL_7C7_TERMINAL_DECISION:
    raise AssertionError('Cell 7C7 terminal PASS mismatch.')
if authorization_7c7.get('authorization_decision') != EXPECTED_CELL_7C7_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C7 authorization decision mismatch.')
if authorization_7c7.get('authorized_cell', {}).get('cell_id') != '7C8':
    raise AssertionError('Cell 7C7 does not authorize Cell 7C8.')
if authorization_7c7.get('answer_key_access_authorized_in_cell_7c8') is not True:
    raise AssertionError('Cell 7C7 did not authorize answer-key access in Cell 7C8.')
if authorization_7c7.get('condition_unblinding_authorized') is not False:
    raise AssertionError('Cell 7C7 unexpectedly authorizes condition unblinding.')
if authorization_7c7.get('scientific_metric_calculation_authorized') is not False:
    raise AssertionError('Cell 7C7 unexpectedly authorizes metric calculation.')
if int(qc_7c7.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C7 QC does not report zero failures.')

structured_path = locate_exact(CELL_7C6_STRUCTURED['filename'], CELL_7C6_STRUCTURED['sha256'])
context_path = locate_exact(CELL_7C4_CONTEXT['filename'], CELL_7C4_CONTEXT['sha256'])
answer_path = locate_exact(CELL_7B3_ANSWER['filename'], CELL_7B3_ANSWER['sha256'])
primary_questions_path = locate_exact(
    CELL_7B3_PRIMARY_QUESTIONS['filename'],
    CELL_7B3_PRIMARY_QUESTIONS['sha256'],
)
rubric_path = locate_exact(CELL_7B3_RUBRIC['filename'], CELL_7B3_RUBRIC['sha256'])
materialization_protocol_path = locate_config_hash(CELL_7B2_MATERIALIZATION_SHA256)
adjudication_protocol_path = locate_config_hash(CELL_7B2_ADJUDICATION_SHA256)

for label, path, expected, source in [
    ('cell_7c6_v2_structured_outputs', structured_path, CELL_7C6_STRUCTURED['sha256'], '7C6_V2'),
    ('cell_7c4_context_inventory', context_path, CELL_7C4_CONTEXT['sha256'], '7C4'),
    ('cell_7b3_structured_answer_keys', answer_path, CELL_7B3_ANSWER['sha256'], '7B3'),
    ('cell_7b3_primary_questions', primary_questions_path, CELL_7B3_PRIMARY_QUESTIONS['sha256'], '7B3'),
    ('cell_7b3_rubric_assignments', rubric_path, CELL_7B3_RUBRIC['sha256'], '7B3'),
    ('cell_7b2_materialization_protocol', materialization_protocol_path, CELL_7B2_MATERIALIZATION_SHA256, '7B2'),
    ('cell_7b2_adjudication_protocol', adjudication_protocol_path, CELL_7B2_ADJUDICATION_SHA256, '7B2'),
]:
    record = verify_exact_artifact(label, path, expected)
    record['source_cell'] = source
    verified_inputs.append(record)

print('Cell 7C7 authorization package       : 5/5 VERIFIED')
print('Cell 7C7 terminal PASS               : VERIFIED')
print('Cell 7C8 authorization               : VERIFIED')
print('Authorized row-level inputs          : 7 exact hashes + sidecars')
print('Condition unblinding                 : NOT AUTHORIZED')
print('Scientific metric calculation        : NOT AUTHORIZED')

Cell 7C7 authorization package       : 5/5 VERIFIED
Cell 7C7 terminal PASS               : VERIFIED
Cell 7C8 authorization               : VERIFIED
Authorized row-level inputs          : 7 exact hashes + sidecars
Condition unblinding                 : NOT AUTHORIZED
Scientific metric calculation        : NOT AUTHORIZED


## 4. Load authorized row content and validate schema/identity invariants

In [5]:
responses = pd.read_parquet(structured_path)
contexts = pd.read_parquet(context_path)
answer_keys = pd.read_parquet(answer_path)
primary_questions = pd.read_csv(
    primary_questions_path,
    dtype={'question_id': 'string', 'question_text': 'string'},
    keep_default_na=False,
    encoding='utf-8',
)
rubrics = pd.read_csv(rubric_path, dtype=str, keep_default_na=False, encoding='utf-8')

materialization_protocol = load_json(materialization_protocol_path)
adjudication_protocol = load_json(adjudication_protocol_path)

REQUIRED_RESPONSE_COLUMNS = {
    'generation_request_id',
    'prompt_instance_id',
    'question_id',
    'blinded_alias',
    'run_id',
    'full_prompt_sha256',
    'response_id',
    'response_status',
    'structured_valid',
    'answer',
    'clinical_significance',
    'conflict_detected',
    'evidence_strength',
    'response_policy',
    'confidence',
    'evidence_ids',
    'reasoning_summary',
}
missing_response = sorted(REQUIRED_RESPONSE_COLUMNS - set(responses.columns))
if missing_response:
    raise AssertionError(
        'Cell 7C6 V2 structured outputs missing required fields: '
        + ', '.join(missing_response)
    )

REQUIRED_CONTEXT_COLUMNS = {
    'question_id',
    'blinded_alias',
    'context_position',
    'packet_id',
    'rcv_accession',
    'context_block',
}
missing_context = sorted(REQUIRED_CONTEXT_COLUMNS - set(contexts.columns))
if missing_context:
    raise AssertionError(
        'Cell 7C4 context inventory missing required fields: '
        + ', '.join(missing_context)
    )

if 'question_id' not in answer_keys.columns:
    raise AssertionError(
        f'Structured answer-key schema is missing question_id. Observed: {list(answer_keys.columns)}'
    )
if 'question_id' not in rubrics.columns:
    raise AssertionError(
        f'Rubric assignment schema is missing question_id. Observed: {list(rubrics.columns)}'
    )

if len(responses) != EXPECTED_RESPONSES:
    raise AssertionError(f'Expected 1,440 responses; observed {len(responses):,}.')
if responses['generation_request_id'].duplicated().any():
    raise AssertionError('Duplicate generation_request_id in structured responses.')
if not responses['structured_valid'].eq(True).all():
    raise AssertionError('At least one Cell 7C6 V2 response is not structured-valid.')

if len(answer_keys) != EXPECTED_QUESTIONS:
    raise AssertionError(f'Expected 80 answer keys; observed {len(answer_keys):,}.')
if answer_keys['question_id'].duplicated().any():
    raise AssertionError('Structured answer keys contain duplicate question_id.')

if not {'question_id', 'question_text'}.issubset(set(primary_questions.columns)):
    raise AssertionError(
        'Primary-question file must contain question_id and question_text. '
        f'Observed: {list(primary_questions.columns)}'
    )
if len(primary_questions) != EXPECTED_QUESTIONS:
    raise AssertionError(
        f'Expected 80 primary questions; observed {len(primary_questions):,}.'
    )
if primary_questions['question_id'].isna().any() or primary_questions['question_id'].duplicated().any():
    raise AssertionError('Primary-question IDs are missing or duplicated.')
if primary_questions['question_text'].isna().any() or primary_questions['question_text'].astype(str).str.strip().eq('').any():
    raise AssertionError('Primary-question text contains missing/blank values.')

if len(rubrics) != EXPECTED_RUBRIC_ROWS:
    raise AssertionError(f'Expected 640 rubric rows; observed {len(rubrics):,}.')
rubric_counts = rubrics.groupby('question_id', dropna=False).size()
if len(rubric_counts) != EXPECTED_QUESTIONS or not rubric_counts.eq(EXPECTED_RUBRICS_PER_QUESTION).all():
    raise AssertionError('Rubric assignment file is not exactly 8 rows × 80 questions.')

if len(contexts) != EXPECTED_CONTEXT_ROWS:
    raise AssertionError(f'Expected 2,400 context rows; observed {len(contexts):,}.')
context_counts = contexts.groupby(['question_id', 'blinded_alias'], dropna=False).size()
if not context_counts.eq(5).all():
    raise AssertionError('Every question/blinded-alias context group must contain exactly 5 rows.')
if not contexts.groupby(['question_id', 'blinded_alias'])['context_position'].apply(
    lambda s: tuple(sorted(int(v) for v in s.tolist())) == (1, 2, 3, 4, 5)
).all():
    raise AssertionError('Context positions are not exactly 1 through 5.')

response_questions = set(responses['question_id'].astype(str))
answer_questions = set(answer_keys['question_id'].astype(str))
primary_question_ids = set(primary_questions['question_id'].astype(str))
rubric_questions = set(rubrics['question_id'].astype(str))
context_questions = set(contexts['question_id'].astype(str))

if response_questions != answer_questions:
    raise AssertionError('Response and answer-key question sets differ.')
if response_questions != primary_question_ids:
    raise AssertionError('Response and primary-question sets differ.')
if response_questions != rubric_questions:
    raise AssertionError('Response and rubric question sets differ.')
if response_questions != context_questions:
    raise AssertionError('Response and context question sets differ.')

question_text_column = 'question_text'

print(f'Structured responses                   : {len(responses):,}')
print(f'Structured answer keys                 : {len(answer_keys):,}')
print(f'Primary questions                      : {len(primary_questions):,}')
print(f'Rubric assignments                     : {len(rubrics):,}')
print(f'Score-blind context rows               : {len(contexts):,}')
print(f'Question-text source                   : primary_questions.{question_text_column}')
print('Question-set identity                  : EXACT ACROSS ALL SOURCES')
print('A-F condition mapping loaded           : NO')
print('Score-bearing artifacts loaded         : NO')

Structured responses                   : 1,440
Structured answer keys                 : 80
Primary questions                      : 80
Rubric assignments                     : 640
Score-blind context rows               : 2,400
Question-text source                   : primary_questions.question_text
Question-set identity                  : EXACT ACROSS ALL SOURCES
A-F condition mapping loaded           : NO
Score-bearing artifacts loaded         : NO


## 5. Materialize opaque review identities, reviewer packet, and internal routing map

In [6]:
def make_review_item_id(generation_request_id: str) -> str:
    return 'REV-' + sha256_text(f'CELL7C8|{generation_request_id}')[:24].upper()

responses = responses.copy()
responses['question_id'] = responses['question_id'].astype(str)
responses['blinded_alias'] = responses['blinded_alias'].astype(str)
responses['generation_request_id'] = responses['generation_request_id'].astype(str)
responses['review_item_id'] = responses['generation_request_id'].map(make_review_item_id)

if responses['review_item_id'].duplicated().any():
    raise AssertionError('Opaque review_item_id collision detected.')

answer_keys = answer_keys.copy()
answer_keys['question_id'] = answer_keys['question_id'].astype(str)

rubrics = rubrics.copy()
rubrics['question_id'] = rubrics['question_id'].astype(str)

contexts = contexts.copy()
contexts['question_id'] = contexts['question_id'].astype(str)
contexts['blinded_alias'] = contexts['blinded_alias'].astype(str)
contexts['context_position'] = contexts['context_position'].astype(int)

answer_key_json_by_question = {
    str(row['question_id']): row_to_json(row)
    for _, row in answer_keys.iterrows()
}
primary_questions = primary_questions.copy()
primary_questions['question_id'] = primary_questions['question_id'].astype(str)

question_text_by_question = {
    str(row['question_id']): str(row['question_text'])
    for _, row in primary_questions.iterrows()
}

rubric_json_by_question = {}
for question_id, group in rubrics.groupby('question_id', sort=False):
    rows = [
        {column: to_native(row[column]) for column in group.columns}
        for _, row in group.iterrows()
    ]
    if len(rows) != 8:
        raise AssertionError(f'Question {question_id} does not have exactly 8 rubric assignments.')
    rubric_json_by_question[str(question_id)] = canonical_json(rows)

context_json_by_question_alias = {}
for (question_id, alias), group in contexts.groupby(
    ['question_id', 'blinded_alias'],
    sort=False,
):
    group = group.sort_values('context_position', kind='mergesort')
    rows = []
    for _, row in group.iterrows():
        rows.append({
            'context_position': int(row['context_position']),
            'packet_id': str(row['packet_id']),
            'rcv_accession': str(row['rcv_accession']),
            'context_block': str(row['context_block']),
        })
    context_json_by_question_alias[(str(question_id), str(alias))] = canonical_json(rows)

RESPONSE_PUBLIC_FIELDS = [
    'answer',
    'clinical_significance',
    'conflict_detected',
    'evidence_strength',
    'response_policy',
    'confidence',
    'evidence_ids',
    'reasoning_summary',
]

review_rows = []
routing_rows = []

for _, row in responses.iterrows():
    qid = str(row['question_id'])
    alias = str(row['blinded_alias'])
    review_item_id = str(row['review_item_id'])

    response_json = canonical_json({
        field: to_native(row[field])
        for field in RESPONSE_PUBLIC_FIELDS
    })

    review_rows.append({
        'review_item_id': review_item_id,
        'question_id': qid,
        'question_text': question_text_by_question[qid],
        'model_response_json': response_json,
        'structured_answer_key_json': answer_key_json_by_question[qid],
        'rubric_bundle_json': rubric_json_by_question[qid],
        'score_blind_context_bundle_json':
            context_json_by_question_alias[(qid, alias)],
    })

    routing_rows.append({
        'review_item_id': review_item_id,
        'generation_request_id': str(row['generation_request_id']),
        'prompt_instance_id': str(row['prompt_instance_id']),
        'question_id': qid,
        'blinded_alias': alias,
        'run_id': int(row['run_id']),
        'full_prompt_sha256': str(row['full_prompt_sha256']),
        'response_id': str(row['response_id']),
    })

review_packet = pd.DataFrame(review_rows).sort_values(
    ['question_id', 'review_item_id'], kind='mergesort'
).reset_index(drop=True)

internal_routing_map = pd.DataFrame(routing_rows).sort_values(
    ['question_id', 'blinded_alias', 'run_id'], kind='mergesort'
).reset_index(drop=True)

if len(review_packet) != 1_440 or review_packet['review_item_id'].duplicated().any():
    raise AssertionError('Blinded review packet identity/count failed.')
if len(internal_routing_map) != 1_440:
    raise AssertionError('Internal routing map count failed.')

for prohibited_column in [
    'generation_request_id', 'prompt_instance_id', 'blinded_alias', 'run_id',
    'condition_id', 'condition_name',
]:
    if prohibited_column in review_packet.columns:
        raise AssertionError(
            f'Reviewer packet exposes prohibited routing/condition field: {prohibited_column}'
        )

print(f'Blinded review items                   : {len(review_packet):,}')
print(f'Internal routing rows                  : {len(internal_routing_map):,}')
print('Reviewer-facing blinded_alias          : ABSENT')
print('Reviewer-facing run_id                 : ABSENT')
print('Reviewer-facing generation_request_id  : ABSENT')
print('A-F condition identity                 : ABSENT')

Blinded review items                   : 1,440
Internal routing rows                  : 1,440
Reviewer-facing blinded_alias          : ABSENT
Reviewer-facing run_id                 : ABSENT
Reviewer-facing generation_request_id  : ABSENT
A-F condition identity                 : ABSENT


## 6. Create two independent reviewer assignments and annotation templates

In [7]:
assignment_rows = []

for reviewer_id in EXPECTED_REVIEWERS:
    order_frame = review_packet[['review_item_id', 'question_id']].copy()
    order_frame['_order_hash'] = order_frame['review_item_id'].map(
        lambda value: sha256_text(f'CELL7C8|{reviewer_id}|{value}')
    )
    order_frame = order_frame.sort_values(
        ['_order_hash', 'review_item_id'],
        kind='mergesort',
    ).reset_index(drop=True)
    order_frame['review_order'] = np.arange(1, len(order_frame) + 1, dtype=np.int32)
    order_frame['reviewer_id'] = reviewer_id
    assignment_rows.extend(
        order_frame[
            ['reviewer_id', 'review_order', 'review_item_id', 'question_id']
        ].to_dict('records')
    )

reviewer_assignments = pd.DataFrame(assignment_rows)

if len(reviewer_assignments) != EXPECTED_REVIEW_ASSIGNMENTS:
    raise AssertionError('Expected exactly 2,880 reviewer assignments.')
if reviewer_assignments.duplicated(['reviewer_id', 'review_item_id']).any():
    raise AssertionError('Duplicate reviewer/item assignment found.')
if not reviewer_assignments.groupby('review_item_id').size().eq(2).all():
    raise AssertionError('Each review item must be assigned to exactly two reviewers.')

rubrics_by_question = {
    qid: group.reset_index(drop=True)
    for qid, group in rubrics.groupby('question_id', sort=False)
}

rubric_rows_out = []
for assignment in reviewer_assignments.itertuples(index=False):
    qid = str(assignment.question_id)
    rubric_group = rubrics_by_question[qid]
    for rubric_index, (_, rubric_row) in enumerate(rubric_group.iterrows(), start=1):
        rubric_rows_out.append({
            'reviewer_id': assignment.reviewer_id,
            'review_order': int(assignment.review_order),
            'review_item_id': assignment.review_item_id,
            'question_id': qid,
            'rubric_index': rubric_index,
            'rubric_assignment_json': row_to_json(rubric_row),
            'reviewer_rubric_score': '',
            'reviewer_rubric_notes': '',
            'review_complete': '',
        })

rubric_scoring_template = pd.DataFrame(rubric_rows_out)

if len(rubric_scoring_template) != EXPECTED_REVIEWER_RUBRIC_ROWS:
    raise AssertionError(
        f'Expected 23,040 reviewer-rubric rows; observed {len(rubric_scoring_template):,}.'
    )

atomic_claim_template = reviewer_assignments.copy()
atomic_claim_template['atomic_claim_annotations_json'] = ''
atomic_claim_template['overall_reviewer_notes'] = ''
atomic_claim_template['review_complete'] = ''

adjudicator_template = pd.DataFrame(columns=[
    'review_item_id',
    'question_id',
    'rubric_index_or_claim_index',
    'disagreement_type',
    'reviewer_1_value',
    'reviewer_2_value',
    'adjudicated_value',
    'adjudicator_notes',
    'adjudication_complete',
])

print(f'Reviewer assignments                  : {len(reviewer_assignments):,}')
print(f'Reviewer-rubric template rows         : {len(rubric_scoring_template):,}')
print(f'Atomic-claim annotation rows          : {len(atomic_claim_template):,}')
print(f'Adjudicator template rows             : {len(adjudicator_template):,} (schema only)')

Reviewer assignments                  : 2,880
Reviewer-rubric template rows         : 23,040
Atomic-claim annotation rows          : 2,880
Adjudicator template rows             : 0 (schema only)


## 7. Reviewer leakage audit and frozen instructions

In [8]:
REVIEWER_FACING_FRAMES = OrderedDict([
    ('review_packet', review_packet),
    ('reviewer_assignments', reviewer_assignments),
    ('rubric_scoring_template', rubric_scoring_template),
    ('atomic_claim_template', atomic_claim_template),
    ('adjudicator_template', adjudicator_template),
])

PROHIBITED_REVIEWER_COLUMN_TOKENS = (
    'blinded_alias',
    'generation_request_id',
    'prompt_instance_id',
    'run_id',
    'condition_id',
    'condition_name',
    'condition_role',
    'full_ges',
    'no_star_ges',
    'combined_metadata',
    'quality_rank',
    'semantic_rank',
    'semantic_score',
    'rrf',
    'p_stable',
    'instability_risk',
)

leaking_columns = {}
for artifact_name, frame in REVIEWER_FACING_FRAMES.items():
    leaks = [
        column for column in frame.columns
        if any(token in column.lower() for token in PROHIBITED_REVIEWER_COLUMN_TOKENS)
    ]
    if leaks:
        leaking_columns[artifact_name] = leaks

if leaking_columns:
    raise AssertionError(f'Reviewer-facing column leakage detected: {leaking_columns}')

FROZEN_BLINDED_ALIASES = [
    'ARM-MICA', 'ARM-ORBIT', 'ARM-KITE',
    'ARM-PULSE', 'ARM-LARCH', 'ARM-NOVA',
]

reviewer_packet_text = '\n'.join(
    review_packet.astype(str).agg(' | '.join, axis=1).tolist()
)
embedded_aliases = [
    alias for alias in FROZEN_BLINDED_ALIASES
    if alias in reviewer_packet_text
]
if embedded_aliases:
    raise AssertionError(
        'Reviewer packet contains blinded alias strings: ' + ', '.join(embedded_aliases)
    )

reviewer_instructions = {
    'cell_id': CELL_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'review_design': {
        'independent_reviewers': EXPECTED_REVIEWERS,
        'third_adjudicator': True,
        'atomic_factual_claims_are_scoring_unit': True,
        'primary_endpoint_definition':
            'fraction of atomic factual claims that are both correct and citation-supported',
        'rubric_dimensions_per_question': 8,
        'egfr_role': 'exploratory; report separately',
    },
    'reviewer_blinding': {
        'condition_identity_hidden': True,
        'blinded_alias_hidden_from_reviewer_packet': True,
        'run_id_hidden_from_reviewer_packet': True,
        'generation_request_id_hidden_from_reviewer_packet': True,
        'GES_and_metadata_scores_hidden': True,
        'quality_and_retrieval_ranks_hidden': True,
        'internal_routing_map_is_not_reviewer_facing': True,
    },
    'reviewer_materials': {
        'review_packet': OUTPUTS['review_packet'].name,
        'reviewer_assignments': OUTPUTS['reviewer_assignments'].name,
        'rubric_scoring_template': OUTPUTS['rubric_scoring_template'].name,
        'atomic_claim_template': OUTPUTS['atomic_claim_template'].name,
    },
    'annotation_rules': {
        'use_frozen_7b2_protocols': True,
        'do_not_modify_model_response_text': True,
        'do_not_modify_answer_key': True,
        'do_not_modify_context_bundle': True,
        'atomic_claim_annotation_field':
            'atomic_claim_annotations_json — reviewer manually records claim-level correctness and citation support following the frozen protocol',
        'rubric_score_field':
            'reviewer_rubric_score — reviewer records the frozen protocol-defined rubric judgment',
    },
    'adjudication': {
        'third_adjudicator_receives_only_disagreements_after_two_independent_reviews': True,
        'adjudicator_template': OUTPUTS['adjudicator_template'].name,
        'no_adjudication_performed_in_cell_7c8': True,
    },
    'scientific_boundary': {
        'condition_unblinding_authorized': False,
        'run_aggregation_authorized': False,
        'scientific_metric_calculation_authorized': False,
        'bootstrap_inference_authorized': False,
        'arm_comparison_authorized': False,
    },
}

print('Reviewer-facing leakage audit            : PASS')
print('Embedded blinded aliases                 : NONE')
print('Condition identity                       : HIDDEN')
print('GES / ranking scores                     : HIDDEN')
print('Internal routing map reviewer-facing     : NO')

Reviewer-facing leakage audit            : PASS
Embedded blinded aliases                 : NONE
Condition identity                       : HIDDEN
GES / ranking scores                     : HIDDEN
Internal routing map reviewer-facing     : NO


## 8. Freeze package, QC, manifest, and terminal boundary

In [9]:
prewrite_checks = OrderedDict([
    ('review_packet_1440', len(review_packet) == 1440),
    ('review_item_ids_unique', review_packet['review_item_id'].nunique() == 1440),
    ('internal_routing_1440', len(internal_routing_map) == 1440),
    ('two_reviewer_assignments_2880', len(reviewer_assignments) == 2880),
    ('two_reviewers_per_item', reviewer_assignments.groupby('review_item_id').size().eq(2).all()),
    ('rubric_template_23040', len(rubric_scoring_template) == 23040),
    ('atomic_claim_template_2880', len(atomic_claim_template) == 2880),
    ('adjudicator_template_empty', len(adjudicator_template) == 0),
    ('answer_keys_80', len(answer_keys) == 80),
    ('primary_questions_80', len(primary_questions) == 80),
    ('rubrics_640', len(rubrics) == 640),
    ('contexts_2400', len(contexts) == 2400),
    ('responses_1440', len(responses) == 1440),
    ('reviewer_column_leakage_zero', len(leaking_columns) == 0),
    ('embedded_alias_leakage_zero', len(embedded_aliases) == 0),
    ('condition_unblinding_not_performed', True),
    ('score_bearing_artifacts_not_loaded', True),
    ('run_aggregation_not_performed', True),
    ('rag_metrics_not_calculated', True),
    ('bootstrap_not_performed', True),
    ('llm_not_called', True),
])

failed = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError('Cell 7C8 prewrite QC failed:\\n- ' + '\\n- '.join(failed))

stable_write_parquet(OUTPUTS['review_packet'], review_packet); write_sidecar(OUTPUTS['review_packet'])
stable_write_csv(OUTPUTS['reviewer_assignments'], reviewer_assignments); write_sidecar(OUTPUTS['reviewer_assignments'])
stable_write_csv(OUTPUTS['rubric_scoring_template'], rubric_scoring_template); write_sidecar(OUTPUTS['rubric_scoring_template'])
stable_write_csv(OUTPUTS['atomic_claim_template'], atomic_claim_template); write_sidecar(OUTPUTS['atomic_claim_template'])
stable_write_csv(OUTPUTS['adjudicator_template'], adjudicator_template); write_sidecar(OUTPUTS['adjudicator_template'])
stable_write_parquet(OUTPUTS['internal_routing_map'], internal_routing_map); write_sidecar(OUTPUTS['internal_routing_map'])
stable_write_json(OUTPUTS['reviewer_instructions'], reviewer_instructions); write_sidecar(OUTPUTS['reviewer_instructions'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['input_inventory'], input_inventory); write_sidecar(OUTPUTS['input_inventory'])

terminal_decision = (
    'PASS_STAGE7C8_1440_OPAQUE_BLINDED_REVIEW_ITEMS_2880_TWO_REVIEWER_ASSIGNMENTS_'
    '23040_RUBRIC_SCORING_ROWS_AND_ATOMIC_CLAIM_TEMPLATES_MATERIALIZED_WITH_INTERNAL_'
    'ROUTING_SEPARATED_CHECKSUM_PROTECTED_NO_CONDITION_UNBLINDING_SCORE_BEARING_'
    'ARTIFACTS_RUN_AGGREGATION_RAG_METRICS_BOOTSTRAP_OR_ARM_COMPARISON_HUMAN_'
    'BLINDED_REVIEW_REQUIRED_NEXT_EXECUTION_NOT_AUTHORIZED'
)

execution_report = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'authorization': {
        'cell_7c7_manifest_sha256': CELL_7C7['manifest']['sha256'],
        'cell_7c7_authorization_sha256': CELL_7C7['authorization']['sha256'],
        'authorization_decision': EXPECTED_CELL_7C7_AUTHORIZATION_DECISION,
    },
    'materialized_counts': {
        'blinded_review_items': int(len(review_packet)),
        'independent_reviewer_assignments': int(len(reviewer_assignments)),
        'reviewer_rubric_rows': int(len(rubric_scoring_template)),
        'reviewer_atomic_claim_rows': int(len(atomic_claim_template)),
        'adjudicator_template_rows_initially': int(len(adjudicator_template)),
        'internal_routing_rows': int(len(internal_routing_map)),
    },
    'reviewer_blinding': {
        'blinded_alias_in_reviewer_artifacts': False,
        'run_id_in_reviewer_artifacts': False,
        'generation_request_id_in_reviewer_artifacts': False,
        'condition_identity_in_reviewer_artifacts': False,
        'GES_or_score_bearing_fields_in_reviewer_artifacts': False,
    },
    'scientific_operations': {
        'answer_key_rows_loaded': True,
        'primary_question_rows_loaded': True,
        'rubric_rows_loaded': True,
        'context_rows_loaded': True,
        'condition_identity_unblinded': False,
        'run_aggregation_performed': False,
        'rag_metrics_calculated': False,
        'bootstrap_inference_performed': False,
        'arm_comparison_performed': False,
        'llm_called': False,
        'adjudication_performed': False,
    },
    'next_required_action': (
        'Two independent human reviewers complete the frozen reviewer templates. '
        'After both reviews, create a separate authorization for disagreement extraction/adjudication import. '
        'Do not unblind conditions or calculate comparative RAG metrics yet.'
    ),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['execution_report'], execution_report); write_sidecar(OUTPUTS['execution_report'])

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'prewrite_checks': {name: bool(value) for name, value in prewrite_checks.items()},
    'passed_checks': len(prewrite_checks),
    'failed_checks': 0,
    'total_checks': len(prewrite_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload); write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c7_manifest_sha256': CELL_7C7['manifest']['sha256'],
        'cell_7c6_v2_structured_outputs_sha256': CELL_7C6_STRUCTURED['sha256'],
        'cell_7b3_structured_answer_keys_sha256': CELL_7B3_ANSWER['sha256'],
        'cell_7b3_primary_questions_sha256': CELL_7B3_PRIMARY_QUESTIONS['sha256'],
        'cell_7b3_rubric_assignments_sha256': CELL_7B3_RUBRIC['sha256'],
        'cell_7c4_context_inventory_sha256': CELL_7C4_CONTEXT['sha256'],
        'cell_7b2_materialization_protocol_sha256': CELL_7B2_MATERIALIZATION_SHA256,
        'cell_7b2_adjudication_protocol_sha256': CELL_7B2_ADJUDICATION_SHA256,
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
            'reviewer_facing': key in {
                'review_packet',
                'reviewer_assignments',
                'rubric_scoring_template',
                'atomic_claim_template',
                'adjudicator_template',
                'reviewer_instructions',
            },
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'internal_routing_map_reviewer_facing': False,
    'condition_unblinding_authorized': False,
    'scientific_metric_calculation_authorized': False,
    'bootstrap_inference_authorized': False,
    'next_authorized_cell': None,
    'next_required_action':
        'Human blinded review by two independent reviewers; later separate adjudication/import authorization.',
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['manifest'], manifest_payload); write_sidecar(OUTPUTS['manifest'])

for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Final Cell 7C8 readback/sidecar failed: {path}')

rb_review = pd.read_parquet(OUTPUTS['review_packet'])
rb_assign = pd.read_csv(OUTPUTS['reviewer_assignments'])
rb_rubric = pd.read_csv(OUTPUTS['rubric_scoring_template'])
rb_claim = pd.read_csv(OUTPUTS['atomic_claim_template'])
rb_route = pd.read_parquet(OUTPUTS['internal_routing_map'])
rb_manifest = load_json(OUTPUTS['manifest'])
rb_qc = load_json(OUTPUTS['qc'])

readback_checks = OrderedDict([
    ('review_packet_1440', len(rb_review) == 1440),
    ('assignments_2880', len(rb_assign) == 2880),
    ('rubric_rows_23040', len(rb_rubric) == 23040),
    ('claim_rows_2880', len(rb_claim) == 2880),
    ('routing_rows_1440', len(rb_route) == 1440),
    ('review_packet_has_no_alias', 'blinded_alias' not in rb_review.columns),
    ('review_packet_has_no_run_id', 'run_id' not in rb_review.columns),
    ('review_packet_has_no_generation_id', 'generation_request_id' not in rb_review.columns),
    ('manifest_next_none', rb_manifest.get('next_authorized_cell') is None),
    ('manifest_unblinding_false', rb_manifest.get('condition_unblinding_authorized') is False),
    ('manifest_metrics_false', rb_manifest.get('scientific_metric_calculation_authorized') is False),
    ('qc_zero_failures', int(rb_qc.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError('Cell 7C8 readback QC failed:\\n- ' + '\\n- '.join(failed_rb))

total_checks = len(prewrite_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C8')
print('BLINDED REVIEWER AND ADJUDICATION PACKET MATERIALIZATION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM AUTHORIZATION')
print(f'Cell 7C7 manifest SHA-256                     : {CELL_7C7["manifest"]["sha256"]}')
print('Cell 7C7 terminal PASS verified               : YES')
print('Cell 7C8 packet-materialization authorization : VERIFIED')

print('\\nBLINDED REVIEW PACKAGE')
print(f'LLM response observations                     : {len(rb_review):,}')
print(f'Primary questions                              : {len(primary_questions):,}')
print(f'Opaque review items                           : {len(rb_review):,}')
print(f'Two-reviewer assignments                      : {len(rb_assign):,}')
print(f'Reviewer-rubric scoring rows                  : {len(rb_rubric):,}')
print(f'Atomic-claim annotation rows                  : {len(rb_claim):,}')
print('Independent reviewers                         : REVIEWER-1 / REVIEWER-2')
print('Third-adjudicator template                    : CREATED')
print('Primary endpoint definition                   : correct AND citation-supported atomic claims')

print('\\nREVIEWER BLINDING')
print('A-F condition identity                        : HIDDEN')
print('Blinded alias                                 : HIDDEN FROM REVIEWERS')
print('Run ID                                        : HIDDEN FROM REVIEWERS')
print('Generation request ID                         : HIDDEN FROM REVIEWERS')
print('GES / metadata / ranking scores               : HIDDEN')
print('Internal routing map                          : SEPARATE — NOT REVIEWER-FACING')

print('\\nCELL 7C8 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nSCIENTIFIC OPERATIONS IN CELL 7C8')
print('Answer keys opened                            : YES — authorized after generation')
print('Rubric rows opened                            : YES — authorized after generation')
print('Condition identities unblinded                : NO')
print('Run aggregation                               : NO')
print('RAG metrics / bootstrap inference             : NO')
print('LLM called                                    : NO')
print('Adjudication performed                        : NO')

print('\\nNEXT BOUNDARY')
print('Next automated cell                           : NOT AUTHORIZED')
print('Required next step                            : two independent human blinded reviews')
print('After reviews                                 : separate disagreement/adjudication import authorization')
print('Condition unblinding / arm comparison         : STILL PROHIBITED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C8
BLINDED REVIEWER AND ADJUDICATION PACKET MATERIALIZATION
Notebook                                      : 15_GES_Aware_Genomic_RAG_Cell_7C8_Blinded_Reviewer_and_Adjudication_Packet_Materialization.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM AUTHORIZATION
Cell 7C7 manifest SHA-256                     : e4dabd9a1eb7715d3f89e44137f369688650f3a812ef6edcfdfef1d7a1be894e
Cell 7C7 terminal PASS verified               : YES
Cell 7C8 packet-materialization authorization : VERIFIED
\nBLINDED REVIEW PACKAGE
LLM response observations                     : 1,440
Primary questions                              : 80
Opaque review items                           : 1,440
Two-reviewer assignments                      : 2,880
Reviewer-rubric scorin